# 02 — Building the compliance auditor

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/04_compliance_auditor/02_build.ipynb)

**Prerequisites**:
- [01_architecture.ipynb](./01_architecture.ipynb)

**What you will learn**:
- How to build a realistic GDPR regulation corpus and parse it into structured requirements
- How to create company policies with intentional compliance gaps
- How to implement a full RAG-based coverage assessor with OpenAI
- How to generate a structured gap report with priorities
- How to produce actionable remediation recommendations
- End-to-end audit pipeline: regulation → gaps → remediations

In [ ]:
# @title Setup — Run this cell first
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "chromadb>=0.4.0" "pandas>=2.0.0"

import os, json, re, textwrap, time
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple
import numpy as np
import pandas as pd

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Clients ──
client: Optional[OpenAI] = None
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM calls will use mock responses")

encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("Sentence encoder loaded ✓")
print("Setup complete ✓")

## Section 1 — Regulation corpus

We create a structured GDPR excerpt covering the articles most commonly audited:
Articles 5-7 (principles, lawfulness, consent), 13-17 (transparency, rights),
25 (data protection by design), and 32 (security).

Each article is parsed into individual requirements with hierarchy preserved.

In [ ]:
GDPR_CORPUS = {
    "Article 5 — Principles relating to processing": [
        ("GDPR-5-1a", "must", "Personal data shall be processed lawfully, fairly and in a transparent manner."),
        ("GDPR-5-1b", "must", "Personal data shall be collected for specified, explicit and legitimate purposes."),
        ("GDPR-5-1c", "must", "Personal data shall be adequate, relevant and limited to what is necessary."),
        ("GDPR-5-1d", "must", "Personal data shall be accurate and kept up to date."),
        ("GDPR-5-1e", "must", "Personal data shall be kept no longer than necessary for the purposes of processing."),
        ("GDPR-5-1f", "must", "Personal data shall be processed with appropriate security including protection against unauthorised processing and accidental loss."),
    ],
    "Article 6 — Lawfulness of processing": [
        ("GDPR-6-1", "must", "Processing shall be lawful only if at least one legal basis applies: consent, contract, legal obligation, vital interests, public interest, or legitimate interests."),
    ],
    "Article 7 — Conditions for consent": [
        ("GDPR-7-1", "must", "The controller shall be able to demonstrate that the data subject has consented to processing."),
        ("GDPR-7-2", "must", "Request for consent shall be clearly distinguishable, intelligible, and in clear and plain language."),
        ("GDPR-7-3", "must", "The data subject shall have the right to withdraw consent at any time."),
    ],
    "Article 13 — Information when data is collected from data subject": [
        ("GDPR-13-1a", "must", "Controller shall provide identity and contact details of the controller at time of collection."),
        ("GDPR-13-1b", "must", "Controller shall provide contact details of the Data Protection Officer."),
        ("GDPR-13-1c", "must", "Controller shall provide the purposes of processing and the legal basis."),
        ("GDPR-13-1d", "must", "Controller shall provide the recipients or categories of recipients of personal data."),
        ("GDPR-13-2a", "must", "Controller shall inform data subject of the data retention period or criteria used to determine it."),
        ("GDPR-13-2b", "must", "Controller shall inform data subject of the right to access, rectification, and erasure."),
    ],
    "Article 15 — Right of access": [
        ("GDPR-15-1", "must", "Data subject shall have the right to obtain confirmation of processing and access to personal data."),
        ("GDPR-15-3", "must", "Controller shall provide a copy of personal data undergoing processing, free of charge."),
    ],
    "Article 16 — Right to rectification": [
        ("GDPR-16-1", "must", "Data subject shall have the right to obtain rectification of inaccurate personal data without undue delay."),
    ],
    "Article 17 — Right to erasure": [
        ("GDPR-17-1", "must", "Data subject shall have the right to obtain erasure of personal data without undue delay."),
        ("GDPR-17-1a", "must", "Erasure applies when data is no longer necessary for the purposes for which it was collected."),
        ("GDPR-17-1b", "must", "Erasure applies when data subject withdraws consent and there is no other legal ground."),
    ],
    "Article 20 — Right to data portability": [
        ("GDPR-20-1", "must", "Data subject shall have the right to receive personal data in a structured, commonly used, machine-readable format."),
        ("GDPR-20-2", "must", "Data subject shall have the right to transmit that data to another controller without hindrance."),
    ],
    "Article 25 — Data protection by design and by default": [
        ("GDPR-25-1", "must", "Controller shall implement appropriate technical and organisational measures for data protection by design."),
        ("GDPR-25-2", "must", "Controller shall ensure that by default only personal data necessary for each specific purpose is processed."),
    ],
    "Article 32 — Security of processing": [
        ("GDPR-32-1a", "must", "Controller shall implement pseudonymisation and encryption of personal data."),
        ("GDPR-32-1b", "must", "Controller shall ensure ongoing confidentiality, integrity, availability and resilience of processing systems."),
        ("GDPR-32-1c", "must", "Controller shall have the ability to restore availability and access to personal data in a timely manner after an incident."),
        ("GDPR-32-1d", "must", "Controller shall implement a process for regularly testing, assessing and evaluating security measures."),
    ],
}

@dataclass
class Requirement:
    """Structured regulatory requirement."""
    id: str
    article: str
    text: str
    obligation: str
    category: str = ""

# Parse into flat list
all_requirements: List[Requirement] = []
for article, reqs in GDPR_CORPUS.items():
    category = article.split("—")[1].strip() if "—" in article else "General"
    for req_id, obligation, text in reqs:
        all_requirements.append(Requirement(
            id=req_id, article=article.split("—")[0].strip(),
            text=text, obligation=obligation, category=category
        ))

print(f"GDPR corpus: {len(GDPR_CORPUS)} articles, {len(all_requirements)} requirements\n")
print("Requirement tree:")
current_art = ""
for r in all_requirements:
    if r.article != current_art:
        current_art = r.article
        print(f"\n  {current_art}")
    print(f"    {r.id:12s} [{r.obligation:5s}] {r.text[:65]}…")

## Section 2 — Company policies

We create three realistic policies for a fictitious SaaS company. Each is ~800 words.

**Intentional gaps** (to exercise the gap detector):
- **No data portability** — Article 20 is unaddressed
- **Weak consent mechanism** — Article 7 only partially covered
- **No DPO contact details** — Article 13(1)(b) missing
- **No breach notification** — Article 33 missing (but we don't audit 33 here)

In [ ]:
PRIVACY_POLICY = """
PRIVACY POLICY — Acme SaaS Corp
Effective Date: January 1, 2024

§1 Introduction
Acme SaaS Corp ("we", "our", "us") provides cloud-based project management software.
This privacy policy describes how we collect, use, store, and protect personal data
of our users in compliance with applicable data protection laws. We are committed
to transparency and responsible data handling across all our services.

§2 Data We Collect
We collect personal data that you provide directly when creating an account: your
full name, email address, company name, job title, and billing information. We also
collect usage data automatically through cookies and analytics tools, including IP
addresses, browser type, pages visited, and session duration. The legal basis for
processing this data is the performance of our contract with you and our legitimate
interest in improving our services.

§3 How We Use Your Data
We use personal data to: (a) provide and maintain our services; (b) process payments
and send invoices; (c) send service-related communications; (d) improve our platform
based on usage patterns; (e) comply with legal obligations. We do not sell personal
data to third parties. We share data only with sub-processors listed in our Data
Processing Agreement.

§4 Data Subject Rights
You have the right to access your personal data by submitting a request to
support@acmesaas.com. We will provide a copy of your data within 30 days of
receiving a verified request. You may request correction of inaccurate data through
your account settings or by contacting support. To delete your account and all
associated data, navigate to Settings > Delete Account. Deletion is processed within
72 hours of confirmation. Backup copies containing your data are purged within 30 days
after account deletion.

§5 Data Collection Transparency
At the time of account creation, we clearly display the types of data we collect and
the purposes for which we process it. Our registration form includes a link to this
privacy policy. We identify ourselves as Acme SaaS Corp, registered at 100 Innovation
Drive, San Francisco, CA 94105.

§6 Consent
By creating an account, you agree to the processing of your data as described in this
policy. You may withdraw your general marketing consent at any time through your
notification preferences. Withdrawal of consent does not affect the lawfulness of
processing performed prior to withdrawal.

§7 Data Sharing and Recipients
We share personal data with the following categories of recipients: (a) cloud hosting
providers (AWS); (b) payment processors (Stripe); (c) analytics providers (Mixpanel);
(d) customer support tools (Zendesk). All sub-processors are bound by Data Processing
Agreements requiring equivalent data protection standards.
"""

DATA_RETENTION_POLICY = """
DATA RETENTION POLICY — Acme SaaS Corp
Effective Date: January 1, 2024

§1 Retention Periods
Active account data is retained for the duration of the customer relationship. Upon
account closure or deletion, personal data is removed from production systems within
72 hours. Backup copies are purged within 30 days of deletion. Financial records and
invoices are retained for 7 years as required by tax law.

§2 Data Minimisation
We collect only the data necessary to provide our services. Registration requires only
name, email, and company name. Additional fields (job title, phone) are optional. We
conduct annual reviews of data fields collected and remove any that are no longer
necessary for service delivery.

§3 Automated Deletion
Inactive accounts (no login for 18 months) receive an automated notification warning
of upcoming deletion. If no action is taken within 30 days, the account and all
associated data are permanently deleted. Users can reactivate their account at any
time before the deletion date by logging in.

§4 Accuracy
Users can update their personal data at any time through the account settings page.
We send periodic reminders (annually) asking users to verify their profile information
is accurate and up to date. Inaccurate data reported by users is corrected within
5 business days.

§5 Lawful Basis Review
We maintain an internal register documenting the lawful basis for each category of
data processing. This register is reviewed quarterly by our legal team to ensure
ongoing compliance. Changes to processing purposes require a new lawful basis
assessment before implementation.
"""

SECURITY_POLICY = """
INFORMATION SECURITY POLICY — Acme SaaS Corp
Effective Date: January 1, 2024

§1 Encryption
All personal data is encrypted at rest using AES-256 encryption. Data in transit is
protected using TLS 1.3. Encryption keys are managed through AWS Key Management
Service and rotated quarterly. Database-level encryption ensures that even raw storage
access does not expose personal data.

§2 Access Control
Access to systems containing personal data is restricted to authorised personnel on a
need-to-know basis. All employees use multi-factor authentication for internal systems.
Role-based access controls ensure that team members can only access data relevant to
their function. Access permissions are reviewed quarterly and revoked upon role change
or departure.

§3 Pseudonymisation
Where possible, we pseudonymise personal data used in development and testing
environments. Production data is never copied to non-production environments without
pseudonymisation. Analytics pipelines use anonymised or aggregated data wherever
feasible.

§4 System Resilience
Our infrastructure runs across multiple AWS availability zones with automatic failover.
Database backups are taken every 6 hours and tested monthly for integrity. Our disaster
recovery plan targets a Recovery Time Objective (RTO) of 4 hours and Recovery Point
Objective (RPO) of 6 hours.

§5 Security Testing
We conduct automated vulnerability scanning weekly using industry-standard tools.
Penetration testing is performed annually by an independent third party. All critical
and high-severity findings are remediated within 30 days. A bug bounty program
provides additional external security review.

§6 Incident Response
Security incidents are classified by severity (Critical, High, Medium, Low). Critical
incidents trigger an immediate response team within 1 hour. All incidents are
documented with root cause analysis. Post-incident reviews are conducted within 48
hours to identify preventive measures.

§7 Privacy by Design
Security and privacy requirements are included in the software development lifecycle
from the design phase. All new features undergo a privacy impact assessment before
development begins. Our engineering team receives annual training on secure coding
practices and data protection principles.
"""

policies = {
    "Privacy Policy": PRIVACY_POLICY,
    "Data Retention Policy": DATA_RETENTION_POLICY,
    "Security Policy": SECURITY_POLICY,
}

for name, text in policies.items():
    word_count = len(text.split())
    section_count = text.count("§")
    print(f"{name:25s} — {word_count:>4} words, {section_count} sections")

print("\nIntentional gaps:")
print("  ✗ No data portability mechanism (Art. 20)")
print("  ✗ Weak consent — no clear opt-in checkbox (Art. 7)")
print("  ✗ No DPO contact details (Art. 13(1)(b))")

## Section 3 — Coverage assessor

Full RAG-based pipeline: for each requirement →
1. Chunk and embed all policies
2. Retrieve top-k policy sections
3. Score coverage 1-5 with LLM
4. Extract supporting evidence
5. Identify gaps

In [ ]:
@dataclass
class PolicyChunk:
    """Indexed chunk of corporate policy."""
    id: str
    source: str
    section: str
    text: str

def chunk_policies(policies: Dict[str, str], max_words: int = 150) -> List[PolicyChunk]:
    """Chunk policies by section, splitting oversized sections."""
    chunks: List[PolicyChunk] = []
    for name, full_text in policies.items():
        sections = re.split(r"(§\d+\s+[^\n]+)", full_text)
        current_section = "Preamble"
        current_text = ""
        for part in sections:
            part = part.strip()
            if not part:
                continue
            if part.startswith("§"):
                # Save previous section
                if current_text.strip():
                    words = current_text.strip().split()
                    for i in range(0, len(words), max_words):
                        chunk_text = " ".join(words[i:i + max_words])
                        cid = f"{name}::{current_section}::c{i // max_words}"
                        chunks.append(PolicyChunk(cid, name, current_section, chunk_text))
                current_section = part
                current_text = ""
            else:
                current_text += " " + part
        # Final section
        if current_text.strip():
            words = current_text.strip().split()
            for i in range(0, len(words), max_words):
                chunk_text = " ".join(words[i:i + max_words])
                cid = f"{name}::{current_section}::c{i // max_words}"
                chunks.append(PolicyChunk(cid, name, current_section, chunk_text))
    return chunks

chunks = chunk_policies(policies)
print(f"Created {len(chunks)} policy chunks\n")

# Embed
chunk_texts = [f"[{c.source} — {c.section}] {c.text}" for c in chunks]
chunk_embeddings = encoder.encode(chunk_texts, show_progress_bar=False)
print(f"Encoded {len(chunk_embeddings)} chunk embeddings")

for c in chunks:
    print(f"  {c.id[:55]:55s} ({len(c.text.split()):>3} words)")

In [ ]:
ASSESSMENT_PROMPT = """You are an expert compliance auditor. Assess how well the
following policy excerpts satisfy the given GDPR requirement.

REQUIREMENT (obligation: {obligation}):
{requirement_text}

RETRIEVED POLICY EXCERPTS:
{excerpts}

Score coverage on this scale:
5 = Fully covered with specific mechanisms and details
4 = Substantially covered, only minor details missing
3 = Partially covered — relevant language exists but significant gaps
2 = Weakly addressed — tangentially related only
1 = Not addressed — no relevant policy language found

Respond in this exact JSON format:
{{
  "score": <int 1-5>,
  "evidence": "<exact quote from policy that best supports coverage>",
  "gap": "<what is missing or insufficient — be specific>",
  "confidence": <float 0.0-1.0>
}}"""

@dataclass
class CoverageResult:
    """Coverage assessment for one requirement."""
    requirement_id: str
    article: str
    requirement_text: str
    score: int
    evidence: str
    gap: str
    confidence: float
    top_chunks: List[str]

def retrieve_chunks(query: str, top_k: int = 5) -> List[Tuple[PolicyChunk, float]]:
    """Retrieve top-k policy chunks for a query."""
    q_emb = encoder.encode([query])
    sims = cosine_similarity(q_emb, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(chunks[i], float(sims[i])) for i in top_idx]

def assess_requirement(req: Requirement) -> CoverageResult:
    """Assess coverage of one requirement."""
    retrieved = retrieve_chunks(req.text, top_k=5)
    excerpts = "\n\n".join(
        f"[Source: {c.source} — {c.section}] (similarity: {sim:.3f})\n{c.text}"
        for c, sim in retrieved
    )
    prompt = ASSESSMENT_PROMPT.format(
        obligation=req.obligation,
        requirement_text=req.text,
        excerpts=excerpts,
    )

    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
    else:
        # Heuristic fallback based on retrieval similarity
        max_sim = max(sim for _, sim in retrieved)
        score = min(5, max(1, int(max_sim * 8)))
        data = {
            "score": score,
            "evidence": retrieved[0][0].text[:100] if retrieved else "",
            "gap": "Mock assessment — set OPENAI_API_KEY for real analysis",
            "confidence": round(float(max_sim), 2),
        }

    return CoverageResult(
        requirement_id=req.id,
        article=req.article,
        requirement_text=req.text,
        score=data["score"],
        evidence=data.get("evidence", ""),
        gap=data.get("gap", ""),
        confidence=data.get("confidence", 0.5),
        top_chunks=[c.id for c, _ in retrieved[:3]],
    )

# ── Process all requirements ──
print(f"Assessing {len(all_requirements)} requirements…\n")
coverage_results: List[CoverageResult] = []
for i, req in enumerate(all_requirements):
    result = assess_requirement(req)
    coverage_results.append(result)
    bar = "█" * result.score + "░" * (5 - result.score)
    print(f"  [{i+1:2d}/{len(all_requirements)}] {req.id:12s} {bar} {result.score}/5  {req.text[:50]}…")
    if not client:
        time.sleep(0.05)  # throttle only for mock

avg_score = np.mean([r.score for r in coverage_results])
print(f"\n✓ All {len(coverage_results)} requirements assessed")
print(f"  Average coverage score: {avg_score:.2f}/5")

## Section 4 — Gap analysis agent

We aggregate coverage results into a structured gap report:
- Overall compliance score per article
- Critical gaps (score < 3)
- Partially covered areas that need reinforcement

In [ ]:
def prioritize(score: int, obligation: str) -> str:
    """Assign priority from score and obligation type."""
    if score <= 1 and obligation == "must":
        return "CRITICAL"
    if score <= 2 and obligation == "must":
        return "HIGH"
    if score <= 2:
        return "MEDIUM"
    if score <= 3:
        return "MEDIUM"
    return "LOW"

@dataclass
class GapItem:
    """A single compliance gap."""
    requirement_id: str
    article: str
    requirement_text: str
    score: int
    gap: str
    priority: str
    evidence: str

# ── Build gap report ──
req_map = {r.id: r for r in all_requirements}
gaps: List[GapItem] = []
for cr in coverage_results:
    req = req_map.get(cr.requirement_id)
    obl = req.obligation if req else "must"
    priority = prioritize(cr.score, obl)
    gaps.append(GapItem(
        requirement_id=cr.requirement_id,
        article=cr.article,
        requirement_text=cr.requirement_text,
        score=cr.score,
        gap=cr.gap,
        priority=priority,
        evidence=cr.evidence,
    ))

# Sort by priority
priority_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
gaps.sort(key=lambda g: (priority_order[g.priority], g.score))

# ── Article-level summary ──
article_scores: Dict[str, List[int]] = {}
for g in gaps:
    article_scores.setdefault(g.article, []).append(g.score)

print("Article-level compliance scores")
print("=" * 60)
for art, scores in article_scores.items():
    avg = np.mean(scores)
    bar = "█" * int(avg) + "░" * (5 - int(avg))
    print(f"  {art:15s}  {bar} {avg:.1f}/5  ({len(scores)} reqs)")

print(f"\nOverall: {np.mean([g.score for g in gaps]):.2f}/5")

# ── Critical and high gaps ──
critical_gaps = [g for g in gaps if g.priority in ("CRITICAL", "HIGH")]
print(f"\nCritical/High gaps: {len(critical_gaps)}")
print("-" * 70)
for g in critical_gaps:
    print(f"\n  [{g.priority:8s}] {g.requirement_id} (score {g.score}/5)")
    print(f"    Requirement: {g.requirement_text[:70]}…")
    print(f"    Gap        : {g.gap[:70]}")

In [ ]:
# ── Full gap report as DataFrame ──
gap_df = pd.DataFrame([asdict(g) for g in gaps])
gap_df = gap_df[["priority", "requirement_id", "article", "score", "gap", "requirement_text"]]
gap_df.columns = ["Priority", "Req ID", "Article", "Score", "Gap Description", "Requirement"]

print("Full gap report (sorted by priority)")
print("=" * 100)
print(gap_df.to_string(index=False, max_colwidth=50))

# Summary stats
print(f"\nSummary:")
print(f"  Total requirements assessed: {len(gaps)}")
print(f"  CRITICAL gaps: {sum(1 for g in gaps if g.priority == 'CRITICAL')}")
print(f"  HIGH gaps    : {sum(1 for g in gaps if g.priority == 'HIGH')}")
print(f"  MEDIUM gaps  : {sum(1 for g in gaps if g.priority == 'MEDIUM')}")
print(f"  LOW / covered: {sum(1 for g in gaps if g.priority == 'LOW')}")

## Section 5 — Remediation generator

For each gap, we generate:
- **Specific policy language** to close the gap
- **Priority** (CRITICAL / HIGH / MEDIUM)
- **Effort estimate** (days)
- **Affected policy** that should be updated

In [ ]:
REMEDIATION_PROMPT = """You are a compliance remediation advisor. For the following
compliance gap, generate specific, actionable remediation.

REQUIREMENT ({obligation}):
{requirement_text}

CURRENT COVERAGE (score {score}/5):
Evidence found: {evidence}
Gap: {gap}

Generate a remediation in this exact JSON format:
{{
  "policy_language": "<specific text to add to the company policy — 2-3 sentences>",
  "affected_policy": "<which policy document should be updated>",
  "action_items": ["<concrete step 1>", "<concrete step 2>", "<concrete step 3>"],
  "effort_days": <int — estimated implementation effort>,
  "rationale": "<why this remediation closes the gap>"
}}"""

@dataclass
class Remediation:
    """Remediation recommendation for a compliance gap."""
    requirement_id: str
    priority: str
    policy_language: str
    affected_policy: str
    action_items: List[str]
    effort_days: int
    rationale: str

def generate_remediation(gap: GapItem) -> Remediation:
    """Generate remediation for a single gap."""
    req = req_map.get(gap.requirement_id)
    obl = req.obligation if req else "must"

    prompt = REMEDIATION_PROMPT.format(
        obligation=obl,
        requirement_text=gap.requirement_text,
        score=gap.score,
        evidence=gap.evidence[:200],
        gap=gap.gap[:200],
    )

    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
    else:
        data = {
            "policy_language": f"Add clause addressing: {gap.requirement_text[:80]}",
            "affected_policy": "Privacy Policy",
            "action_items": [
                f"Draft policy language for {gap.requirement_id}",
                "Review with legal counsel",
                "Update policy document and publish",
            ],
            "effort_days": max(1, 6 - gap.score),
            "rationale": f"Mock remediation — set OPENAI_API_KEY for detailed advice",
        }

    return Remediation(
        requirement_id=gap.requirement_id,
        priority=gap.priority,
        policy_language=data.get("policy_language", ""),
        affected_policy=data.get("affected_policy", "Unknown"),
        action_items=data.get("action_items", []),
        effort_days=data.get("effort_days", 3),
        rationale=data.get("rationale", ""),
    )

# ── Generate for top 5 critical/high gaps ──
top_gaps = [g for g in gaps if g.priority in ("CRITICAL", "HIGH")][:5]
if len(top_gaps) < 5:
    top_gaps += [g for g in gaps if g.priority == "MEDIUM"][:5 - len(top_gaps)]

print(f"Generating remediations for top {len(top_gaps)} gaps…\n")
remediations: List[Remediation] = []
for g in top_gaps:
    rem = generate_remediation(g)
    remediations.append(rem)
    print(f"[{rem.priority:8s}] {rem.requirement_id}")
    print(f"  Policy  : {rem.affected_policy}")
    print(f"  Language: {rem.policy_language[:80]}…")
    print(f"  Effort  : {rem.effort_days} days")
    print(f"  Actions : {len(rem.action_items)} steps")
    for a in rem.action_items:
        print(f"    • {a[:70]}")
    print()

## Section 6 — End-to-end audit demo

Full pipeline: GDPR requirements → policy retrieval → coverage scoring → gap analysis
→ remediation generation. We produce both an **executive summary** and a **detailed report**.

In [ ]:
# ── Executive summary ──
total_reqs = len(all_requirements)
avg = np.mean([r.score for r in coverage_results])
critical_count = sum(1 for g in gaps if g.priority == "CRITICAL")
high_count = sum(1 for g in gaps if g.priority == "HIGH")
fully_covered = sum(1 for r in coverage_results if r.score >= 4)
total_effort = sum(r.effort_days for r in remediations)

print("╔" + "═" * 58 + "╗")
print("║       COMPLIANCE AUDIT — EXECUTIVE SUMMARY              ║")
print("╠" + "═" * 58 + "╣")
print(f"║  Framework        : GDPR (EU 2016/679)                  ║")
print(f"║  Requirements     : {total_reqs:>3}                                  ║")
print(f"║  Avg coverage     : {avg:>4.2f} / 5.00                          ║")
print(f"║  Fully covered    : {fully_covered:>3} / {total_reqs}  ({fully_covered/total_reqs*100:.0f}%)                      ║")
print(f"║  CRITICAL gaps    : {critical_count:>3}                                  ║")
print(f"║  HIGH gaps        : {high_count:>3}                                  ║")
print(f"║  Est. remediation : {total_effort:>3} person-days                      ║")
print("╚" + "═" * 58 + "╝")

In [ ]:
# ── Detailed report by article ──
print("\nDETAILED COMPLIANCE REPORT")
print("=" * 80)

for article in GDPR_CORPUS:
    art_key = article.split("—")[0].strip()
    art_results = [r for r in coverage_results if r.article == art_key]
    if not art_results:
        continue
    art_avg = np.mean([r.score for r in art_results])
    status = "✓ PASS" if art_avg >= 4 else ("⚠ PARTIAL" if art_avg >= 2.5 else "✗ FAIL")

    print(f"\n{'─' * 80}")
    print(f"  {article}")
    print(f"  Score: {art_avg:.1f}/5  Status: {status}")
    print(f"{'─' * 80}")

    for r in art_results:
        bar = "█" * r.score + "░" * (5 - r.score)
        print(f"    {r.requirement_id:12s} {bar} {r.score}/5")
        if r.score < 4:
            print(f"      Gap: {r.gap[:65]}")

# ── Remediation roadmap ──
print(f"\n\n{'=' * 80}")
print("REMEDIATION ROADMAP")
print(f"{'=' * 80}")
for i, rem in enumerate(remediations, 1):
    print(f"\n  {i}. [{rem.priority}] {rem.requirement_id} — {rem.affected_policy}")
    print(f"     {rem.policy_language[:75]}…")
    print(f"     Effort: {rem.effort_days} days")

print(f"\n  Total estimated effort: {total_effort} person-days")
print(f"  Estimated cost (at $200/hr): ${total_effort * 8 * 200:,.0f}")
print(f"  vs. manual audit cost      : $50,000 – $500,000")

## Exercises

1. **Add a fourth policy** — Create a "Consent Management Policy" (~400 words)
   that explicitly addresses GDPR Articles 6 and 7. Re-run the pipeline and observe
   how scores change for those requirements.

2. **Multi-framework audit** — Add five SOC 2 CC-series requirements to the corpus.
   Run the same policies through both GDPR and SOC 2 assessment. Which gaps overlap?

3. **Retrieval tuning** — Experiment with `top_k = 3` vs `top_k = 7` in
   `retrieve_chunks`. Does retrieving more context improve or degrade LLM scoring?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | A structured regulation corpus with 30+ requirements enables systematic auditing |
| 2 | Intentional gaps in policies (portability, consent) are reliably detected |
| 3 | RAG-based assessment combines retrieval precision with LLM judgement |
| 4 | Gap reports must be prioritised: CRITICAL/must violations first |
| 5 | Remediation should include specific policy language, not just generic advice |
| 6 | End-to-end pipeline runs in minutes vs. weeks for manual audit |

## What's Next

In **03 — Evaluate**, we rigorously measure the auditor's accuracy: extraction
completeness, scoring calibration, false gap rate, and remediation quality.

---

## References

1. European Parliament, *Regulation (EU) 2016/679 — General Data Protection Regulation*, 2016.
2. OpenAI, *GPT-4o-mini API Documentation*, 2024. — <https://platform.openai.com/docs/>
3. Sentence-Transformers — <https://www.sbert.net/>
4. GDPR Enforcement Tracker — <https://www.enforcementtracker.com/>